Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    _create_pipeline,
    StatsModelsEstimator,
    OptunaOptimizer,
    smote_objective,
    pure_smote_objective,
    GridSearchOptimizer,
)

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-smote_default-model'
add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
n_trials = 50

## Optimization functions
Optuna will not be used further, except first try, it is enough to conduct grid search

In [5]:
def optimize_smote_optuna(
    target:str,
    estimator:object,
    estimator_params:dict,
    n_trials:int,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=True,
):
    """
    Optimize the SMOTE parameters for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        n_trials: The number of trials for optimization. Defaults to n_trials.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics
    )

    opt = OptunaOptimizer(
        objective=partial(smote_objective, ml_pipe=ml_pipe),
        study_name="smote_study",
        direction="maximize",
    )

    opt.optimize(n_trials=n_trials)

    best_params = opt.study.best_params
    print('best_params')
    display(best_params)

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=best_params,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )
    
    
def optimize_smote_grid(
    target:str,
    estimator:object,
    estimator_params:dict,
    param_grid:dict=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=True,
):
    """
    Optimize the SMOTE parameters with GridSearchOptimizer for a given estimator.

    Args:
        target: The target variable for the model.
        estimator: The machine learning estimator to be optimized.
        estimator_params: Parameters for the estimator.
        model_postfix: A postfix to append to the model name. Defaults to model_postfix.
        add_smote: Whether to add SMOTE to the pipeline. Defaults to add_smote.
        is_smotenc: Whether to use SMOTENC for categorical features. Defaults to is_smotenc.
        smote_params: Parameters for SMOTE. Defaults to smote_params.
        scoring_metrics: Metrics for scoring the model. Defaults to scoring_metrics.
        save_model_and_metrics: Whether to save the model and metrics. Defaults to save_model_and_metrics.
    """
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=smote_params,
        scoring_metrics=scoring_metrics
    )
    
    if param_grid is None:
        param_grid = {
        'sampling_strategy': [0.6, 0.7, 0.8, 0.9, 1.0],
        'k_neighbors': list(range(3,11)),
    }
    opt = GridSearchOptimizer(
        objective=partial(pure_smote_objective, ml_pipe=ml_pipe),
        param_grid=param_grid,
        verbose=True,
    )

    opt.optimize()
    best_params = opt.best_params

    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=best_params,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [ ]:
estimator_class = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    n_trials=n_trials,
    save_model_and_metrics=False,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-11 16:46:23,656] A new study created in memory with name: smote_study
[I 2025-04-11 16:46:23,727] Trial 0 finished with value: 0.7795918367346939 and parameters: {'k_neighbors': 5, 'sampling_strategy': 0.6}. Best is trial 0 with value: 0.7795918367346939.
[I 2025-04-11 16:46:23,795] Trial 1 finished with value: 0.7826336975273146 and parameters: {'k_neighbors': 3, 'sampling_strategy': 1.0}. Best is trial 1 with value: 0.7826336975273146.
[I 2025-04-11 16:46:23,863] Trial 2 finished with value: 0.800925925925926 and parameters: {'k_neighbors': 9, 'sampling_strategy': 1.0}. Best is trial 2 with value: 0.800925925925926.
[I 2025-04-11 16:46:23,931] Trial 3 finished with value: 0.7826336975273146 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.7}. Best is trial 2 with value: 0.800925925925926.
[I 2025-04-11 16:46:23,999] Trial 4 finished with value: 0.8055555555555556 and parameters: {'k_neighbors': 6, 'sampling_strategy': 0.6}. Best is trial 4 with value: 0.8055555555

best_params


...

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.883488
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.763393
cv_test_accuracy_balanced_median,0.77709
cv_test_roc_auc_median,0.874613


In [7]:
estimator_class = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   0%|          | 0/40 [00:00<?, ?it/s]

Progress: 1/40.	Score: 0.7795918367346939.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.883488
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.763393
cv_test_accuracy_balanced_median,0.77709
cv_test_roc_auc_median,0.874613


Model saved in ../results/models_modelling4/LogisticRegression_splashing_smote_opt-smote_default-model


Full grid search can find better solution, than Optuna with small number of iterations - like 40! It appears in some Random_seed

In [8]:
estimator_class = LogisticRegression
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   5%|▌         | 2/40 [00:00<00:02, 16.47it/s]

Progress: 1/40.	Score: 0.8179068360556563.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.
Progress: 2/40.	Score: 0.8179068360556563.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.7}.
Progress: 3/40.	Score: 0.8179068360556563.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.8}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.785539
holdout_test_accuracy_balanced,0.825
holdout_test_roc_auc,0.95
holdout_test_f1,0.708333
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.826797
cv_test_accuracy_balanced_median,0.869231
cv_test_roc_auc_median,0.946886


Model saved in ../results/models_modelling4/LogisticRegression_no_fragmentation_smote_opt-smote_default-model


# KNClassifier

In [9]:
estimator_class = KNeighborsClassifier
target = 'splashing'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▎         | 1/40 [00:00<00:06,  5.90it/s]

Progress: 1/40.	Score: 0.8089668615984406.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smote_opt-smote...
holdout_test_f1_macro,0.810348
holdout_test_accuracy_balanced,0.80787
holdout_test_roc_auc,0.874228
holdout_test_f1,0.865979
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.835913
cv_test_accuracy_balanced_median,0.844427
cv_test_roc_auc_median,0.918731


Model saved in ../results/models_modelling4/KNeighborsClassifier_splashing_smote_opt-smote_default-model


In [10]:
estimator_class = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   5%|▌         | 2/40 [00:00<00:03, 11.39it/s]

Progress: 1/40.	Score: 0.8110483364720653.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.
Progress: 2/40.	Score: 0.8110483364720653.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.7}.
Progress: 3/40.	Score: 0.7881773399014779.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.8}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smote_op...
holdout_test_f1_macro,0.780518
holdout_test_accuracy_balanced,0.809091
holdout_test_roc_auc,0.937273
holdout_test_f1,0.695652
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.854396
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.928571


Model saved in ../results/models_modelling4/KNeighborsClassifier_no_fragmentation_smote_opt-smote_default-model


# SVC

In [11]:
estimator_class = SVC
target = 'splashing'
estimator_params = {
    'probability': True,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▎         | 1/40 [00:00<00:06,  5.90it/s]

Progress: 1/40.	Score: 0.8313725490196078.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smote_opt-smote_default-model
holdout_test_f1_macro,0.793956
holdout_test_accuracy_balanced,0.789352
holdout_test_roc_auc,0.893519
holdout_test_f1,0.857143
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.863049
cv_test_accuracy_balanced_median,0.885449
cv_test_roc_auc_median,0.922601


Model saved in ../results/models_modelling4/SVC_splashing_smote_opt-smote_default-model


In [12]:
estimator_class = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

Optimizing parameters:   2%|▎         | 1/40 [00:00<00:06,  6.27it/s]

Progress: 1/40.	Score: 0.8503207412687099.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smote_opt-smote_default-m...
holdout_test_f1_macro,0.857143
holdout_test_accuracy_balanced,0.886364
holdout_test_roc_auc,0.95
holdout_test_f1,0.8
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.850704
cv_test_accuracy_balanced_median,0.877289
cv_test_roc_auc_median,0.947802


Model saved in ../results/models_modelling4/SVC_no_fragmentation_smote_opt-smote_default-model


# CatBoost

In [13]:
estimator_class = CatBoostClassifier
target = 'splashing'
estimator_params = {
    'verbose': False,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▎         | 1/40 [00:03<02:33,  3.93s/it]

Progress: 1/40.	Score: 0.8940886699507389.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "CatBoostClassifier"


,0
target,splashing
model,CatBoostClassifier_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.85
holdout_test_accuracy_balanced,0.83912
holdout_test_roc_auc,0.946759
holdout_test_f1,0.9
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.860788
cv_test_accuracy_balanced_median,0.873839
cv_test_roc_auc_median,0.95356


Model saved in ../results/models_modelling4/CatBoostClassifier_splashing_smote_opt-smote_default-model


In [14]:
estimator_class = CatBoostClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': False,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "CatBoostClassifier"


,0
target,no_fragmentation
model,CatBoostClassifier_no_fragmentation_smotenc_de...
holdout_test_f1_macro,0.916089
holdout_test_accuracy_balanced,0.922727
holdout_test_roc_auc,0.984545
holdout_test_f1,0.878049
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.881326
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.97265


Model saved in ../results/models_modelling4/CatBoostClassifier_no_fragmentation_smotenc_default


# XGBoost

In [15]:
estimator_class = XGBClassifier
target = 'splashing'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "XGBClassifier"


,0
target,splashing
model,XGBClassifier_splashing_smotenc_default
holdout_test_f1_macro,0.88
holdout_test_accuracy_balanced,0.868056
holdout_test_roc_auc,0.932485
holdout_test_f1,0.92
holdout_test_accuracy,0.893333
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.946032


Model saved in ../results/models_modelling4/XGBClassifier_splashing_smotenc_default


In [16]:
estimator_class = XGBClassifier
target = 'no_fragmentation'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "XGBClassifier"


,0
target,no_fragmentation
model,XGBClassifier_no_fragmentation_smotenc_default
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.988182
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.877289
cv_test_roc_auc_median,0.967033


Model saved in ../results/models_modelling4/XGBClassifier_no_fragmentation_smotenc_default


# AdaBoost

In [14]:
estimator_class = AdaBoostClassifier
target = 'splashing'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▎         | 1/40 [00:00<00:12,  3.18it/s]

Progress: 1/40.	Score: 0.8635477582846004.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "AdaBoostClassifier"


,0
target,splashing
model,AdaBoostClassifier_splashing_smote_opt-smote_d...
holdout_test_f1_macro,0.751994
holdout_test_accuracy_balanced,0.75
holdout_test_roc_auc,0.854552
holdout_test_f1,0.824742
holdout_test_accuracy,0.773333
cv_test_f1_macro_median,0.794892
cv_test_accuracy_balanced_median,0.794892
cv_test_roc_auc_median,0.891641


Model saved in ../results/models_modelling4/AdaBoostClassifier_splashing_smote_opt-smote_default-model


In [18]:
estimator_class = AdaBoostClassifier
target = 'no_fragmentation'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "AdaBoostClassifier"


,0
target,no_fragmentation
model,AdaBoostClassifier_no_fragmentation_smotenc_de...
holdout_test_f1_macro,0.839194
holdout_test_accuracy_balanced,0.861364
holdout_test_roc_auc,0.951364
holdout_test_f1,0.772727
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.826067
cv_test_accuracy_balanced_median,0.823077
cv_test_roc_auc_median,0.930403


Model saved in ../results/models_modelling4/AdaBoostClassifier_no_fragmentation_smotenc_default


# Random Forest

In [15]:
estimator_class = RandomForestClassifier
target = 'splashing'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▎         | 1/40 [00:00<00:18,  2.14it/s]

Progress: 1/40.	Score: 0.8464285714285715.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "RandomForestClassifier"


,0
target,splashing
model,RandomForestClassifier_splashing_smote_opt-smo...
holdout_test_f1_macro,0.82
holdout_test_accuracy_balanced,0.810185
holdout_test_roc_auc,0.943673
holdout_test_f1,0.88
holdout_test_accuracy,0.84
cv_test_f1_macro_median,0.876935
cv_test_accuracy_balanced_median,0.876935
cv_test_roc_auc_median,0.947619


Model saved in ../results/models_modelling4/RandomForestClassifier_splashing_smote_opt-smote_default-model


In [20]:
estimator_class = RandomForestClassifier
target = 'no_fragmentation'
estimator_params = {}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "RandomForestClassifier"


,0
target,no_fragmentation
model,RandomForestClassifier_no_fragmentation_smoten...
holdout_test_f1_macro,0.885894
holdout_test_accuracy_balanced,0.904545
holdout_test_roc_auc,0.99
holdout_test_f1,0.837209
holdout_test_accuracy,0.906667
cv_test_f1_macro_median,0.875762
cv_test_accuracy_balanced_median,0.867216
cv_test_roc_auc_median,0.954701


Model saved in ../results/models_modelling4/RandomForestClassifier_no_fragmentation_smotenc_default


# LightGBM

In [7]:
estimator_class = LGBMClassifier
target = 'splashing'
estimator_params = {
    'verbose': -1,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

Optimizing parameters:   2%|▎         | 1/40 [00:01<00:50,  1.30s/it]

Progress: 1/40.	Score: 0.8770726129216695.	Considered params: {'k_neighbors': 3, 'sampling_strategy': 0.6}.


...

Progress: 27/40.	Score: 0.8683385579937304.	Considered params: {'k_neighbors': 8, 'sampling_strategy': 0.7}.


Optimizing parameters:  70%|███████   | 28/40 [00:36<00:15,  1.33s/it]

Progress: 28/40.	Score: 0.8928571428571428.	Considered params: {'k_neighbors': 8, 'sampling_strategy': 0.8}.


Optimizing parameters:  70%|███████   | 28/40 [00:37<00:16,  1.35s/it]


KeyboardInterrupt: 

In [22]:
estimator_class = LGBMClassifier
target = 'no_fragmentation'
estimator_params = {
    'verbose': -1,
}

estimator = estimator_class(
    **estimator_params
)

optimize_smote_grid(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[LightGBM] [Info] Number of positive: 234, number of negative: 234
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000350 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 746
[LightGBM] [Info] Number of data points in the train set: 468, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with 

,0
target,no_fragmentation
model,LGBMClassifier_no_fragmentation_smotenc_default
holdout_test_f1_macro,0.916089
holdout_test_accuracy_balanced,0.922727
holdout_test_roc_auc,0.986364
holdout_test_f1,0.878049
holdout_test_accuracy,0.933333
cv_test_f1_macro_median,0.886022
cv_test_accuracy_balanced_median,0.90293
cv_test_roc_auc_median,0.974359


Model saved in ../results/models_modelling4/LGBMClassifier_no_fragmentation_smotenc_default


# For the notebook with Model optimization!

In [ ]:
# def update_estimator_params(
#     ml_pipe:MLPipeline,
#     suggested_params:dict,
# ) -> dict:
#     """Update the estimator parameters based on the pipeline parameters.

#     Args:
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.
#         suggested_params: A dictionary containing the suggested Estimator hyperparameters.
    
#     Returns:
#         A dictionary containing the estimator parameters.
#     """
#     estimator_params = ml_pipe._pipeline_params['estimator_params']
#     estimator_params.update(suggested_params)
#     return estimator_params

# def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
#     """Objective function for logistic regression optimization using Optuna.

#     Args:
#         trial: An Optuna trial object used to suggest hyperparameters.
#         ml_pipe: An instance of MLPipeline used for model training and evaluation.

#     Returns:
#         The score of the model based on the specified evaluation metric.
#     """
    
#     categorical_features = ('wettability', 'inclination')
    
#     random_state = ml_pipe._pipeline_params['random_state']
    
#     # sample params
#     C = trial.suggest_loguniform('C', 1e-4, 1e3)
    
#     # SMOTE params
#     add_smote = trial.suggest_categorical('add_smote', [True, False])
#     if add_smote:
#         is_smotenc = trial.suggest_categorical('is_smotenc', [True, False])
#         smote_params = {
#             'sampling_strategy': trial.suggest_categorical(
#                 'sampling_strategy', [0.6, 0.7, 0.8, 0.9, 1.0]
#             ),
#             'k_neighbors': trial.suggest_int('k_neighbors', 3, 10),
#             'random_state': random_state,
#         }
#     else:
#         is_smotenc = False
#         smote_params = None
#     if is_smotenc:
#         wettability_cat = trial.suggest_categorical('wettability_cat', [True, False])
#         if wettability_cat:
#             inclination_cat = trial.suggest_categorical('inclination_cat', [True, False])
#         else:
#             inclination_cat = True
        
        
#         mask = [wettability_cat, inclination_cat]
        
#         masked_features = [
#             feature for feature, mask_value in zip(categorical_features, mask) 
#             if mask_value
#         ]
#         smote_params = {
#             **smote_params,
#             'categorical_features': masked_features,
#         }

#     suggested_params = {
#         "C": C,
#     }
    
#     estimator_params = update_estimator_params(ml_pipe, suggested_params)

#     estimator = LogisticRegression(
#         **estimator_params,
#         random_state=random_state,
#     )

#     score = ml_pipe.step(
#         estimator=estimator,
#         add_smote=add_smote,
#         is_smotenc=is_smotenc,
#         smote_params=smote_params,
#     )
    
#     return score



# opt = OptunaOptimizer(
#     objective=partial(logreg_objective, ml_pipe=ml_pipe),
#     study_name="logreg_study",
#     direction="maximize",
# )

# opt.optimize(n_trials=200)

# best_params = opt.study.best_trial.params
# display(best_params)
# # estimator_params = update_estimator_params(ml_pipe, best_params)

# # estimator = LogisticRegression(
# #     **estimator_params,
# #     random_state=ml_pipe._pipeline_params['random_state'],
# # )

[I 2025-04-11 00:02:50,034] A new study created in memory with name: logreg_study
[I 2025-04-11 00:02:50,105] Trial 0 finished with value: 0.7754010695187166 and parameters: {'C': 7.659422407680541, 'add_smote': False}. Best is trial 0 with value: 0.7754010695187166.
[I 2025-04-11 00:02:50,166] Trial 1 finished with value: 0.7730205278592375 and parameters: {'C': 0.3986291498738802, 'add_smote': False}. Best is trial 0 with value: 0.7754010695187166.
[I 2025-04-11 00:02:50,237] Trial 2 finished with value: 0.7787307032590051 and parameters: {'C': 0.0030108676374517194, 'add_smote': True, 'is_smotenc': False, 'sampling_strategy': 0.6, 'k_neighbors': 5}. Best is trial 2 with value: 0.7787307032590051.
[I 2025-04-11 00:02:50,569] Trial 3 finished with value: 0.800925925925926 and parameters: {'C': 17.34448264461045, 'add_smote': True, 'is_smotenc': True, 'sampling_strategy': 0.9, 'k_neighbors': 4, 'wettability_cat': True, 'inclination_cat': False}. Best is trial 3 with value: 0.8009259259

{'C': 1.6850403782968544,
 'add_smote': True,
 'is_smotenc': False,
 'sampling_strategy': 0.7,
 'k_neighbors': 4}